# 06 - Lakebase analyst review queue (step 7)

Scores recent transfers with the model version currently served by `transfer-fraud-model` and loads the top-scored ones into a Lakebase (Postgres) review queue that the Streamlit app reads and writes.

* **Where:** Lakebase project `fraud-rt-online`, database `databricks_postgres`, schema `fraud_review` (separate from the feature-store online tables in database `mlo_serverless_stable_catalog`).
* **Tables:** `review_queue` (one row per flagged transfer: score, threshold, model version, status `pending/approved/rejected/escalated`, reviewer, notes) and `review_actions` (append-only audit log of analyst actions).
* **Scoring:** `fe.score_batch` with point-in-time feature lookups from the offline Feature Views, so each transfer is scored with the features it had at transfer time. The endpoint uses online features (the latest window), which is right for new transfers but wrong for re-scoring past ones.
* **Flag rule:** the top `flag_top_pct` (default 2%) of scores in the lookback window, matching the precision/recall at top 2% metric from `05_train_model`. The threshold is stored on each row.
* **Upsert rule:** re-runs refresh scores only for rows still `pending`; analyst decisions are never overwritten.

In [0]:
%pip install -q "databricks-feature-engineering>=0.17" lightgbm "psycopg[binary]" "databricks-sdk>=0.118.0"
dbutils.library.restartPython()

In [0]:
import time
import pandas as pd
import psycopg
from pyspark.sql import functions as F
from databricks.sdk import WorkspaceClient
from databricks.sdk.service.serving import EndpointStateConfigUpdate
from databricks.feature_engineering import FeatureEngineeringClient

dbutils.widgets.text("catalog", "mlo_serverless_stable_catalog")
dbutils.widgets.text("schema", "fraud_rt")
dbutils.widgets.text("lookback_days", "7")
dbutils.widgets.text("flag_top_pct", "2")
CATALOG, SCHEMA = dbutils.widgets.get("catalog"), dbutils.widgets.get("schema")
LOOKBACK_DAYS = int(dbutils.widgets.get("lookback_days"))
FLAG_TOP_PCT = float(dbutils.widgets.get("flag_top_pct"))
MODEL_NAME = f"{CATALOG}.{SCHEMA}.transfer_fraud_model"
ENDPOINT = "transfer-fraud-model"
LAKEBASE_EP = "projects/fraud-rt-online/branches/production/endpoints/primary"
PG_DB = "databricks_postgres"

w = WorkspaceClient()
fe = FeatureEngineeringClient()

# Queue scores must come from the same model version the endpoint serves; wait out an in-flight config update.
for _ in range(90):
    ep = w.serving_endpoints.get(ENDPOINT)
    if ep.state.config_update != EndpointStateConfigUpdate.IN_PROGRESS:
        break
    time.sleep(20)
assert ep.state.config_update != EndpointStateConfigUpdate.IN_PROGRESS, "endpoint still updating"
served = [e for e in ep.config.served_entities if e.entity_name == MODEL_NAME]
assert len(served) == 1, served
MODEL_VERSION = served[0].entity_version
MODEL_URI = f"models:/{MODEL_NAME}/{MODEL_VERSION}"
print(MODEL_URI, "| lookback", LOOKBACK_DAYS, "days | flag top", FLAG_TOP_PCT, "%")

In [0]:
PG_HOST = w.postgres.get_endpoint(name=LAKEBASE_EP).status.hosts.host
PG_USER = w.current_user.me().user_name

def pg_connect():
    # Short-lived OAuth token (1h); minted per connection, never stored.
    token = w.postgres.generate_database_credential(endpoint=LAKEBASE_EP).token
    return psycopg.connect(host=PG_HOST, dbname=PG_DB, user=PG_USER, password=token, sslmode="require")

DDL = """
CREATE SCHEMA IF NOT EXISTS fraud_review;
CREATE TABLE IF NOT EXISTS fraud_review.review_queue (
  transfer_id     text PRIMARY KEY,
  customer_id     text NOT NULL,
  corridor        text NOT NULL,
  payout_method   text,
  amount_usd      numeric(12,2),
  transfer_ts     timestamptz NOT NULL,
  fraud_score     double precision NOT NULL CHECK (fraud_score BETWEEN 0 AND 1),
  score_threshold double precision NOT NULL,
  model_name      text NOT NULL,
  model_version   text NOT NULL,
  scored_at       timestamptz NOT NULL DEFAULT now(),
  status          text NOT NULL DEFAULT 'pending' CHECK (status IN ('pending','approved','rejected','escalated')),
  reviewer        text,
  reviewed_at     timestamptz,
  review_notes    text,
  updated_at      timestamptz NOT NULL DEFAULT now()
);
CREATE INDEX IF NOT EXISTS review_queue_status_score_idx ON fraud_review.review_queue (status, fraud_score DESC);
CREATE INDEX IF NOT EXISTS review_queue_customer_idx ON fraud_review.review_queue (customer_id);
CREATE TABLE IF NOT EXISTS fraud_review.review_actions (
  action_id    bigserial PRIMARY KEY,
  transfer_id  text NOT NULL REFERENCES fraud_review.review_queue(transfer_id),
  old_status   text,
  new_status   text NOT NULL,
  reviewer     text NOT NULL,
  notes        text,
  acted_at     timestamptz NOT NULL DEFAULT now()
);
COMMENT ON TABLE fraud_review.review_queue IS 'Analyst review queue: transfers flagged by transfer_fraud_model (score >= threshold). Synthetic data.';
COMMENT ON TABLE fraud_review.review_actions IS 'Append-only audit log of analyst approve/reject/escalate actions.';
"""
with pg_connect() as conn:
    conn.execute(DDL)
    print(conn.execute("""SELECT table_name, count(*) FROM information_schema.columns
                           WHERE table_schema = 'fraud_review' GROUP BY table_name ORDER BY 1""").fetchall())

In [0]:
recent = (spark.table(f"{CATALOG}.{SCHEMA}.silver_transfers")
          .where(F.col("ts") >= F.expr(f"(SELECT max(ts) FROM {CATALOG}.{SCHEMA}.silver_transfers) - INTERVAL {LOOKBACK_DAYS} DAYS"))
          .select("transfer_id", "customer_id", "corridor", "ts", F.col("ts").alias("event_ts"),
                  "payout_method", "amount_usd", "is_fraud"))

# score_batch looks up each Feature View as of the row's timestamps, like the training set.
scored = fe.score_batch(model_uri=MODEL_URI, df=recent, result_type="double")
spd = scored.select("transfer_id", "customer_id", "corridor", "payout_method", "amount_usd", "ts", "is_fraud",
                    F.col("prediction").alias("fraud_score")).toPandas()
# Known generator issue: live runs reuse the fixed seed, so some transfer_ids repeat at different times
# (distinct sequence numbers). The queue is keyed by transfer_id, so keep the latest occurrence of each.
n_before = len(spd)
spd = spd.sort_values("ts").drop_duplicates("transfer_id", keep="last").reset_index(drop=True)
print(f"dropped {n_before - len(spd)} older rows that reuse a transfer_id")
assert spd["fraud_score"].between(0, 1).all(), "model must return probabilities"
assert spd["fraud_score"].nunique() > 2, "scores look like 0/1 labels, not probabilities"
print(len(spd), "transfers scored |", spd.ts.min(), "->", spd.ts.max())
print(spd["fraud_score"].describe().round(4).to_string())

In [0]:
THRESHOLD = float(spd["fraud_score"].quantile(1 - FLAG_TOP_PCT / 100))
flagged = spd[spd["fraud_score"] >= THRESHOLD].sort_values("fraud_score", ascending=False).copy()
# Sanity check only: the served model was refit on ALL transfers, so this window is in-sample (see 05 for holdout metrics).
hits, total_fraud = int(flagged.is_fraud.sum()), int(spd.is_fraud.sum())
print(f"threshold={THRESHOLD:.4f} flagged={len(flagged)} of {len(spd)} "
      f"| in-sample label check: {hits} fraud in queue ({hits/len(flagged):.1%} precision), "
      f"{hits}/{total_fraud} fraud in window caught ({hits/max(total_fraud,1):.1%} recall)")
display(flagged.head(10))

In [0]:
UPSERT = """
INSERT INTO fraud_review.review_queue
  (transfer_id, customer_id, corridor, payout_method, amount_usd, transfer_ts,
   fraud_score, score_threshold, model_name, model_version)
VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s, %s)
ON CONFLICT (transfer_id) DO UPDATE SET
  customer_id = EXCLUDED.customer_id, corridor = EXCLUDED.corridor, payout_method = EXCLUDED.payout_method,
  amount_usd = EXCLUDED.amount_usd, transfer_ts = EXCLUDED.transfer_ts,
  fraud_score = EXCLUDED.fraud_score, score_threshold = EXCLUDED.score_threshold,
  model_version = EXCLUDED.model_version, scored_at = now(), updated_at = now()
WHERE fraud_review.review_queue.status = 'pending'
"""
# is_fraud is the synthetic label: it is NOT written to the queue, so analysts don't see the answer.
rows = [(r.transfer_id, r.customer_id, r.corridor, r.payout_method, float(r.amount_usd),
         pd.Timestamp(r.ts).tz_localize("UTC").to_pydatetime(), float(r.fraud_score), THRESHOLD, MODEL_NAME, str(MODEL_VERSION))
        for r in flagged.itertuples()]
with pg_connect() as conn:
    with conn.cursor() as cur:
        cur.executemany(UPSERT, rows)
print("upserted", len(rows), "rows")

In [0]:
with pg_connect() as conn:
    q = pd.DataFrame(conn.execute("""SELECT transfer_id, fraud_score, model_version, status, transfer_ts FROM fraud_review.review_queue""").fetchall(),
                     columns=["transfer_id", "fraud_score", "model_version", "status", "transfer_ts"])
    by_status = conn.execute("SELECT status, count(*) FROM fraud_review.review_queue GROUP BY status ORDER BY 1").fetchall()

m = flagged[["transfer_id", "fraud_score"]].merge(q, on="transfer_id", how="left", suffixes=("_spark", "_pg"))
missing = int(m["fraud_score_pg"].isna().sum())
pending = m[m.status == "pending"]
max_diff = float((pending.fraud_score_spark - pending.fraud_score_pg).abs().max()) if len(pending) else 0.0
stale = int((~q.transfer_id.isin(flagged.transfer_id)).sum())  # rows from earlier runs no longer in the top set
print("queue rows by status:", by_status, "| rows from earlier runs not flagged now:", stale)
m = m.merge(flagged[["transfer_id", "ts"]], on="transfer_id")
pending = m[m.status == "pending"]
ts_mismatch = int((pending.ts.dt.tz_localize("UTC") != pd.to_datetime(pending.transfer_ts, utc=True)).sum())
print("pending rows whose transfer_ts differs from this run:", ts_mismatch)
print(f"flagged this run: {len(flagged)} | missing from queue: {missing} | max score diff on pending rows: {max_diff:.2e}")
assert missing == 0 and max_diff < 1e-9 and ts_mismatch == 0